# Donut Fine-Tuning on FUNSD: Document Extraction via Distillation

**Goal:** Fine-tune Donut (200M params) to extract structured key-value pairs from noisy scanned documents.

**Method:** LoRA fine-tuning (freeze base weights, train ~2-4M adapter params)

**Dataset:** FUNSD (149 train / 50 test) + Augraphy augmentation

**Hardware:** Google Colab free T4 GPU (16GB VRAM)

**Expected time:** ~2-3 hours training

---

## How This Works

1. **Donut architecture:** Swin Transformer (vision encoder) → BART (text decoder)
2. **Pre-trained on:** 1.6M synthetic documents (SynthDoG) — already knows how to read documents
3. **LoRA:** We freeze the 200M pre-trained weights and train small adapter matrices (~2-4M params)
4. **Input:** Document image (PNG)
5. **Output:** Structured JSON with extracted key-value pairs
6. **No OCR needed** — Donut goes directly from pixels to text

## Step 0: Setup & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi
print("\n" + "="*60)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
%%capture
# Install dependencies
!pip install transformers[torch] datasets peft accelerate
!pip install sentencepiece protobuf
!pip install augraphy
!pip install Pillow
!pip install evaluate jiwer  # for metrics

In [ ]:
import json
import os
import re
import random
import numpy as np
from pathlib import Path
from typing import Dict, List, Any

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from transformers import (
    VisionEncoderDecoderModel,
    VisionEncoderDecoderConfig,
    DonutProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    GenerationConfig,
)
from peft import LoraConfig, get_peft_model, TaskType

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("All imports successful.")

## Step 1: Upload & Load Data

Upload the following from your `chaos-eval` repo:
- `data/donut_format/training.json`
- `data/donut_format/testing.json`
- `data/funsd/dataset/` (the full FUNSD image folders)

Or clone the repo directly:

In [ ]:
# Option A: Clone repo (includes data prep scripts)
# !git clone https://github.com/MyatPyaePaingPie/chaos-eval.git
# %cd chaos-eval

# Option B: Upload FUNSD directly
# Download FUNSD: https://guillaumejaume.github.io/FUNSD/
# Then upload and extract:
# !unzip funsd.zip -d data/funsd/

# Option C: Download FUNSD programmatically
!mkdir -p data/funsd data/donut_format

import urllib.request
import zipfile

FUNSD_URL = "https://guillaumejaume.github.io/FUNSD/dataset.zip"
zip_path = "data/funsd/dataset.zip"

if not os.path.exists("data/funsd/dataset"):
    print("Downloading FUNSD dataset...")
    urllib.request.urlretrieve(FUNSD_URL, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall("data/funsd/")
    os.remove(zip_path)
    print("Done.")
else:
    print("FUNSD already exists.")

# Verify
train_imgs = list(Path("data/funsd/dataset/training_data/images").glob("*.png"))
test_imgs = list(Path("data/funsd/dataset/testing_data/images").glob("*.png"))
print(f"Training images: {len(train_imgs)}")
print(f"Testing images: {len(test_imgs)}")

In [ ]:
# ============================================================
# Parse FUNSD annotations into structured key-value pairs
# ============================================================

from collections import defaultdict

def parse_funsd_annotation(annotation_path: str) -> dict:
    """Extract key-value pairs from FUNSD annotation by following question→answer links."""
    with open(annotation_path, 'r') as f:
        data = json.load(f)

    elements = {item['id']: item for item in data['form']}

    # Build link map: question_id → set of answer_ids
    link_map = defaultdict(set)
    for item in data['form']:
        for link in item.get('linking', []):
            from_id, to_id = link
            from_elem = elements.get(from_id)
            to_elem = elements.get(to_id)
            if from_elem and to_elem:
                if from_elem['label'] == 'question' and to_elem['label'] == 'answer':
                    link_map[from_id].add(to_id)
                elif from_elem['label'] == 'answer' and to_elem['label'] == 'question':
                    link_map[to_id].add(from_id)
                elif from_elem['label'] == 'header' and to_elem['label'] == 'answer':
                    link_map[from_id].add(to_id)

    kv_pairs = {}
    for q_id, a_ids in link_map.items():
        question = elements[q_id]
        q_text = question['text'].strip().rstrip(':').strip()
        if not q_text or q_text == ':':
            continue
        answers = []
        for a_id in a_ids:
            a_text = elements[a_id]['text'].strip()
            if a_text:
                answers.append(a_text)
        if answers:
            kv_pairs[q_text] = '; '.join(answers) if len(answers) > 1 else answers[0]

    return kv_pairs


def prepare_split(split: str) -> list:
    """Prepare a FUNSD split for Donut training."""
    split_dir = 'training_data' if split == 'training' else 'testing_data'
    ann_dir = Path(f"data/funsd/dataset/{split_dir}/annotations")
    img_dir = Path(f"data/funsd/dataset/{split_dir}/images")

    dataset = []
    for ann_file in sorted(ann_dir.glob('*.json')):
        doc_id = ann_file.stem
        img_file = img_dir / f'{doc_id}.png'
        if not img_file.exists():
            continue
        kv_pairs = parse_funsd_annotation(str(ann_file))
        if not kv_pairs:  # skip blank templates
            continue
        dataset.append({
            "image": str(img_file),
            "ground_truth": json.dumps({"gt_parse": kv_pairs}, ensure_ascii=False),
            "doc_id": doc_id,
        })
    return dataset


train_data = prepare_split('training')
test_data = prepare_split('testing')

print(f"Training samples: {len(train_data)}")
print(f"Testing samples:  {len(test_data)}")
print(f"\nSample ground truth:")
gt = json.loads(train_data[0]['ground_truth'])
for k, v in list(gt['gt_parse'].items())[:3]:
    print(f"  {k}: {v}")

## Step 2: Data Augmentation with Augraphy

149 training docs is small. We'll use Augraphy to create realistic degraded variants:
- Noise (Gaussian, salt & pepper)
- Blur
- JPEG artifacts
- Paper aging
- Ink effects

Target: 5x augmentation → ~735 training samples

In [ ]:
import augraphy

def create_augmentation_pipeline(intensity: str = 'medium'):
    """
    Create document-specific augmentation pipeline.

    intensity: 'light', 'medium', 'heavy'
    """
    if intensity == 'light':
        return augraphy.AugraphyPipeline(
            ink_phase=[
                augraphy.Brightness(brightness_range=(0.9, 1.1), p=0.5),
            ],
            paper_phase=[
                augraphy.NoiseTexturize(sigma_range=(1, 3), turbulence_range=(1, 3), p=0.3),
            ],
            post_phase=[
                augraphy.Jpeg(quality_range=(70, 95), p=0.5),
            ],
        )
    elif intensity == 'medium':
        return augraphy.AugraphyPipeline(
            ink_phase=[
                augraphy.Brightness(brightness_range=(0.8, 1.2), p=0.5),
                augraphy.BleedThrough(intensity_range=(0.05, 0.15), p=0.3),
            ],
            paper_phase=[
                augraphy.NoiseTexturize(sigma_range=(2, 5), turbulence_range=(2, 5), p=0.5),
            ],
            post_phase=[
                augraphy.Jpeg(quality_range=(50, 85), p=0.5),
                augraphy.GaussianBlur(blur_kernel_size=(3, 3), p=0.3),
            ],
        )
    else:  # heavy
        return augraphy.AugraphyPipeline(
            ink_phase=[
                augraphy.Brightness(brightness_range=(0.6, 1.4), p=0.7),
                augraphy.BleedThrough(intensity_range=(0.1, 0.3), p=0.5),
                augraphy.Letterpress(n_copies=1, p=0.3),
            ],
            paper_phase=[
                augraphy.NoiseTexturize(sigma_range=(3, 8), turbulence_range=(3, 6), p=0.7),
            ],
            post_phase=[
                augraphy.Jpeg(quality_range=(30, 70), p=0.7),
                augraphy.GaussianBlur(blur_kernel_size=(3, 5), p=0.5),
            ],
        )


def augment_dataset(dataset: list, augmentations_per_sample: int = 4) -> list:
    """
    Augment dataset with Augraphy.
    Creates augmentations_per_sample variants of each document.
    Original images are kept as-is.
    """
    aug_dir = Path("data/augmented")
    aug_dir.mkdir(exist_ok=True)

    pipelines = {
        'light': create_augmentation_pipeline('light'),
        'medium': create_augmentation_pipeline('medium'),
        'heavy': create_augmentation_pipeline('heavy'),
    }

    # Distribution: 2 light, 1 medium, 1 heavy
    aug_schedule = ['light', 'light', 'medium', 'heavy']

    augmented = list(dataset)  # start with originals
    total = len(dataset)

    for i, sample in enumerate(dataset):
        if (i + 1) % 25 == 0:
            print(f"  Augmenting {i+1}/{total}...")

        img = np.array(Image.open(sample['image']).convert('RGB'))

        for j, intensity in enumerate(aug_schedule[:augmentations_per_sample]):
            try:
                aug_img = pipelines[intensity](img)
                aug_path = aug_dir / f"{sample['doc_id']}_aug{j}_{intensity}.png"
                Image.fromarray(aug_img).save(str(aug_path))

                augmented.append({
                    "image": str(aug_path),
                    "ground_truth": sample['ground_truth'],  # same labels
                    "doc_id": f"{sample['doc_id']}_aug{j}",
                })
            except Exception as e:
                pass  # skip failed augmentations

    return augmented


print(f"Original training samples: {len(train_data)}")
print("Augmenting (this takes a few minutes)...")
train_augmented = augment_dataset(train_data, augmentations_per_sample=4)
print(f"Augmented training samples: {len(train_augmented)}")
print(f"Augmentation ratio: {len(train_augmented)/len(train_data):.1f}x")

## Step 3: Load Donut Model & Processor

In [ ]:
# ============================================================
# Load pre-trained Donut
# ============================================================

MODEL_ID = "naver-clova-ix/donut-base"

print("Loading Donut processor...")
processor = DonutProcessor.from_pretrained(MODEL_ID)

print("Loading Donut model...")
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)

print(f"\nModel loaded:")
print(f"  Encoder: {model.encoder.__class__.__name__}")
print(f"  Decoder: {model.decoder.__class__.__name__}")
print(f"  Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ============================================================
# Add special tokens for our task
# ============================================================

# Donut uses task-specific tokens to know what to extract
TASK_START_TOKEN = "<s_funsd>"
TASK_END_TOKEN = "</s_funsd>"

# Add tokens to tokenizer
new_tokens = [TASK_START_TOKEN, TASK_END_TOKEN]
processor.tokenizer.add_tokens(new_tokens)

# Resize model embeddings to match new tokenizer
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Set model config
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(TASK_START_TOKEN)

print(f"Tokenizer vocab size: {len(processor.tokenizer)}")
print(f"Task start token ID: {model.config.decoder_start_token_id}")

In [ ]:
# ============================================================
# Apply LoRA
# ============================================================

# LoRA config: only train small adapter matrices
# This prevents catastrophic forgetting of pre-trained knowledge
lora_config = LoraConfig(
    r=16,                          # rank of adapter matrices
    lora_alpha=32,                 # scaling factor
    lora_dropout=0.1,              # dropout for regularization
    target_modules=[               # which layers to adapt
        "q_proj", "v_proj",        # attention queries and values
        "k_proj", "out_proj",      # attention keys and output
    ],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, lora_config)

# Print trainable params
model.print_trainable_parameters()
# Expected: ~1-3% of total params are trainable

## Step 4: Create PyTorch Dataset

In [ ]:
# ============================================================
# Dataset class for Donut
# ============================================================

MAX_LENGTH = 768  # max output tokens
IMAGE_SIZE = [1280, 960]  # Donut's expected input size


class FUNSDDonutDataset(Dataset):
    """
    Dataset for Donut fine-tuning on FUNSD.

    Each sample:
    - pixel_values: processed image tensor
    - labels: tokenized target JSON string
    """

    def __init__(self, data: list, processor: DonutProcessor, max_length: int = MAX_LENGTH):
        self.data = data
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        # Load and process image
        image = Image.open(sample['image']).convert('RGB')
        pixel_values = self.processor(
            image,
            return_tensors="pt"
        ).pixel_values.squeeze()

        # Build target sequence: <s_funsd> {json} </s_funsd>
        gt = json.loads(sample['ground_truth'])
        target_text = (
            TASK_START_TOKEN
            + json.dumps(gt['gt_parse'], ensure_ascii=False)
            + TASK_END_TOKEN
        )

        # Tokenize target
        labels = self.processor.tokenizer(
            target_text,
            add_special_tokens=False,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze()

        # Replace padding with -100 so loss ignores it
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels,
        }


# Create datasets
train_dataset = FUNSDDonutDataset(train_augmented, processor)
test_dataset = FUNSDDonutDataset(test_data, processor)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Test dataset:  {len(test_dataset)} samples")

# Verify a sample
sample = train_dataset[0]
print(f"\nSample shapes:")
print(f"  pixel_values: {sample['pixel_values'].shape}")
print(f"  labels: {sample['labels'].shape}")
print(f"  non-padding tokens: {(sample['labels'] != -100).sum().item()}")

## Step 5: Training

In [ ]:
# ============================================================
# Training configuration
# ============================================================

OUTPUT_DIR = "./donut-funsd-lora"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training hyperparameters
    num_train_epochs=5,
    per_device_train_batch_size=2,        # T4 can handle 2 with LoRA
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,         # effective batch size = 8
    learning_rate=3e-4,                    # higher LR for LoRA
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,                             # mixed precision

    # Evaluation
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Generation (for evaluation)
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,

    # Logging
    logging_dir="./logs",
    logging_steps=25,
    report_to="none",                      # no wandb/tensorboard

    # Misc
    remove_unused_columns=False,
    dataloader_pin_memory=True,
    seed=SEED,
)

print("Training config:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

In [ ]:
# ============================================================
# Initialize Trainer & Start Training
# ============================================================

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

print("Starting training...")
print(f"Total training steps: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * int(training_args.num_train_epochs)}")
print("="*60)

train_result = trainer.train()

print("\n" + "="*60)
print("Training complete!")
print(f"  Train loss: {train_result.training_loss:.4f}")
print(f"  Train time: {train_result.metrics.get('train_runtime', 0)/60:.1f} minutes")

In [ ]:
# Save the LoRA adapter
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# Check adapter size
adapter_size = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
    if os.path.isfile(os.path.join(OUTPUT_DIR, f))
) / (1024 * 1024)
print(f"Adapter size: {adapter_size:.1f} MB")

## Step 6: Evaluation

Run inference on all 50 test documents and measure:
1. **Field accuracy** — how many key-value pairs match ground truth
2. **JSON validity** — does the model output parseable JSON
3. **Hallucination rate** — does it fabricate fields not in the document
4. **Speed** — inference time per document

In [ ]:
# ============================================================
# Inference function
# ============================================================

import time

def extract_from_image(image_path: str, model, processor) -> dict:
    """
    Run Donut inference on a single document image.
    Returns extracted key-value pairs and metadata.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    # Load and process image
    image = Image.open(image_path).convert('RGB')
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

    # Generate
    start_time = time.time()

    task_prompt = TASK_START_TOKEN
    decoder_input_ids = processor.tokenizer(
        task_prompt,
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=MAX_LENGTH,
            num_beams=3,
            early_stopping=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.convert_tokens_to_ids(TASK_END_TOKEN),
        )

    inference_time = time.time() - start_time

    # Decode output
    raw_output = processor.tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Extract JSON between task tokens
    json_str = raw_output
    json_str = json_str.replace(TASK_START_TOKEN, "").replace(TASK_END_TOKEN, "")
    json_str = json_str.replace("<s>", "").replace("</s>", "").replace("<pad>", "")
    json_str = json_str.strip()

    # Try to parse JSON
    try:
        extracted = json.loads(json_str)
        json_valid = True
    except json.JSONDecodeError:
        extracted = {"_raw": json_str}
        json_valid = False

    return {
        "extracted": extracted,
        "json_valid": json_valid,
        "raw_output": raw_output,
        "inference_time": inference_time,
    }


# Quick test on first image
print("Testing inference on first test image...")
result = extract_from_image(test_data[0]['image'], model, processor)
print(f"  JSON valid: {result['json_valid']}")
print(f"  Inference time: {result['inference_time']:.2f}s")
print(f"  Extracted: {json.dumps(result['extracted'], indent=2)[:500]}")

In [ ]:
# ============================================================
# Full evaluation on test set
# ============================================================

def normalize_text(text: str) -> str:
    """Normalize text for comparison."""
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)  # collapse whitespace
    text = re.sub(r'[^a-z0-9\s]', '', text)  # remove punctuation
    return text


def evaluate_extraction(predicted: dict, ground_truth: dict) -> dict:
    """Compare predicted vs ground truth key-value pairs."""
    gt_keys = set(k for k in ground_truth.keys() if not k.startswith('_'))
    pred_keys = set(k for k in predicted.keys() if not k.startswith('_'))

    # Key matching (fuzzy - normalized)
    gt_norm = {normalize_text(k): v for k, v in ground_truth.items() if not k.startswith('_')}
    pred_norm = {normalize_text(k): v for k, v in predicted.items() if not k.startswith('_')}

    matched_keys = set(gt_norm.keys()) & set(pred_norm.keys())

    # Value accuracy for matched keys
    correct_values = 0
    for key in matched_keys:
        if normalize_text(str(gt_norm[key])) == normalize_text(str(pred_norm[key])):
            correct_values += 1

    # Hallucinated keys (in prediction but not in ground truth)
    hallucinated = pred_norm.keys() - gt_norm.keys()

    # Missing keys (in ground truth but not in prediction)
    missing = gt_norm.keys() - pred_norm.keys()

    total_gt = len(gt_norm)

    return {
        "total_gt_fields": total_gt,
        "total_pred_fields": len(pred_norm),
        "matched_keys": len(matched_keys),
        "correct_values": correct_values,
        "key_recall": len(matched_keys) / total_gt if total_gt > 0 else 0,
        "value_accuracy": correct_values / len(matched_keys) if matched_keys else 0,
        "field_accuracy": correct_values / total_gt if total_gt > 0 else 0,
        "hallucinated_keys": len(hallucinated),
        "missing_keys": len(missing),
    }


# Run evaluation on all test documents
print("Evaluating on test set...")
print("="*60)

results = []
total_time = 0

for i, sample in enumerate(test_data):
    gt = json.loads(sample['ground_truth'])['gt_parse']

    # Run inference
    output = extract_from_image(sample['image'], model, processor)
    total_time += output['inference_time']

    # Evaluate
    metrics = evaluate_extraction(output['extracted'], gt)
    metrics['doc_id'] = sample['doc_id']
    metrics['json_valid'] = output['json_valid']
    metrics['inference_time'] = output['inference_time']
    results.append(metrics)

    if (i + 1) % 10 == 0:
        avg_acc = np.mean([r['field_accuracy'] for r in results])
        print(f"  [{i+1}/{len(test_data)}] Running avg accuracy: {avg_acc:.1%}")

print(f"\nDone. Total inference time: {total_time:.1f}s")

In [ ]:
# ============================================================
# Results Summary
# ============================================================

json_valid_count = sum(1 for r in results if r['json_valid'])
avg_field_accuracy = np.mean([r['field_accuracy'] for r in results])
avg_key_recall = np.mean([r['key_recall'] for r in results])
avg_value_accuracy = np.mean([r['value_accuracy'] for r in results if r['matched_keys'] > 0])
total_hallucinated = sum(r['hallucinated_keys'] for r in results)
avg_inference_time = np.mean([r['inference_time'] for r in results])

print("\n" + "="*60)
print("  DONUT FINE-TUNING RESULTS (FUNSD Test Set)")
print("="*60)
print(f"  Documents tested:     {len(results)}")
print(f"  JSON validity:        {json_valid_count}/{len(results)} ({json_valid_count/len(results):.0%})")
print(f"  Field accuracy:       {avg_field_accuracy:.1%}")
print(f"  Key recall:           {avg_key_recall:.1%}")
print(f"  Value accuracy:       {avg_value_accuracy:.1%}")
print(f"  Total hallucinations: {total_hallucinated}")
print(f"  Avg inference time:   {avg_inference_time:.2f}s")
print("="*60)

print("\n--- Comparison with Claude Baseline ---")
print(f"{'Metric':<25} {'Claude':>10} {'Donut':>10} {'Delta':>10}")
print("-"*55)
print(f"{'Field accuracy':<25} {'98.5%':>10} {f'{avg_field_accuracy:.1%}':>10} {f'{avg_field_accuracy - 0.985:+.1%}':>10}")
print(f"{'Hallucination rate':<25} {'0%':>10} {f'{total_hallucinated/sum(r["total_gt_fields"] for r in results):.1%}':>10}")
print(f"{'Inference time/doc':<25} {'~3s':>10} {f'{avg_inference_time:.2f}s':>10}")
print(f"{'Cost per 1K pages':<25} {'~$30':>10} {'~$0.50':>10}")
print(f"{'Model size':<25} {'>100GB':>10} {'~200MB':>10}")

In [ ]:
# ============================================================
# Per-document breakdown
# ============================================================

print("\nPer-Document Results:")
print(f"{'Doc ID':<25} {'Fields':>8} {'Acc':>8} {'Hallu':>8} {'JSON':>8} {'Time':>8}")
print("-"*65)

for r in sorted(results, key=lambda x: x['field_accuracy'], reverse=True):
    print(
        f"{r['doc_id']:<25} "
        f"{r['total_gt_fields']:>8} "
        f"{r['field_accuracy']:>7.0%} "
        f"{r['hallucinated_keys']:>8} "
        f"{'Yes' if r['json_valid'] else 'NO':>8} "
        f"{r['inference_time']:>7.2f}s"
    )

## Step 7: Export for CPU Deployment

In [ ]:
# ============================================================
# Merge LoRA weights back into base model for deployment
# ============================================================

print("Merging LoRA weights into base model...")
merged_model = model.merge_and_unload()

MERGED_DIR = "./donut-funsd-merged"
merged_model.save_pretrained(MERGED_DIR)
processor.save_pretrained(MERGED_DIR)

# Check merged model size
total_size = 0
for root, dirs, files in os.walk(MERGED_DIR):
    for f in files:
        total_size += os.path.getsize(os.path.join(root, f))
print(f"Merged model size: {total_size / (1024**2):.0f} MB")
print(f"Saved to: {MERGED_DIR}")

In [ ]:
# ============================================================
# Test CPU inference with merged model
# ============================================================

print("Testing CPU inference...")

# Force CPU
cpu_model = VisionEncoderDecoderModel.from_pretrained(MERGED_DIR).to("cpu")
cpu_processor = DonutProcessor.from_pretrained(MERGED_DIR)

# Run on first test image
image = Image.open(test_data[0]['image']).convert('RGB')
pixel_values = cpu_processor(image, return_tensors="pt").pixel_values

task_prompt = TASK_START_TOKEN
decoder_input_ids = cpu_processor.tokenizer(
    task_prompt, add_special_tokens=False, return_tensors="pt"
).input_ids

start = time.time()
with torch.no_grad():
    outputs = cpu_model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=MAX_LENGTH,
        num_beams=1,  # greedy for speed
        pad_token_id=cpu_processor.tokenizer.pad_token_id,
        eos_token_id=cpu_processor.tokenizer.convert_tokens_to_ids(TASK_END_TOKEN),
    )
cpu_time = time.time() - start

output_text = cpu_processor.tokenizer.decode(outputs[0], skip_special_tokens=False)
print(f"CPU inference time: {cpu_time:.2f}s")
print(f"Output: {output_text[:300]}")

In [ ]:
# ============================================================
# Download model for local use
# ============================================================

# Zip the merged model
!cd {MERGED_DIR} && zip -r ../donut-funsd-merged.zip .
print(f"\nDownload: donut-funsd-merged.zip")
print("This model runs on CPU — no GPU required for inference.")

# Also save results
with open("evaluation_results.json", "w") as f:
    json.dump({
        "summary": {
            "documents_tested": len(results),
            "json_validity": json_valid_count / len(results),
            "field_accuracy": avg_field_accuracy,
            "key_recall": avg_key_recall,
            "value_accuracy": avg_value_accuracy,
            "total_hallucinations": total_hallucinated,
            "avg_inference_time_gpu": avg_inference_time,
            "cpu_inference_time": cpu_time,
        },
        "per_document": results,
    }, f, indent=2)
print("Results saved to evaluation_results.json")

## Summary

### What We Did
1. Parsed FUNSD annotations into structured key-value pairs
2. Augmented 149 training docs → ~735 with Augraphy noise
3. Fine-tuned Donut-base with LoRA (only ~2% of params trained)
4. Evaluated on 50 unseen test documents
5. Exported merged model for CPU deployment

### Key Numbers
| Metric | Claude Baseline | Donut Fine-tuned |
|--------|----------------|------------------|
| Field accuracy | 98.5% | See results above |
| Hallucination rate | 0% | See results above |
| Inference time | ~3s/doc | See results above |
| Cost per 1K pages | ~$30 | ~$0.50 (self-hosted) |
| Model size | >100GB (API) | ~800MB (local) |

### Next Steps
- **If accuracy is low (<70%):** Add SROIE + CORD datasets, increase augmentation
- **If hallucination is high:** Add confidence thresholds, increase LoRA rank
- **If speed matters:** Export to ONNX + INT8 quantization → ~200MB, <1s on CPU
- **For production:** Implement the hybrid architecture (Donut fast path + LLM fallback)